# Caution ! this is a "Third Party Tool" which aims to modify gamefiles.
## this will VOID your WARRANTY
### proceed with caution

the following executions are my journey into editing Timberborn savefiles. i will go over some fundamentals although most information can be read and understood from the files themselves. 

But first i would like to express that editing in this way should only be done once you are already well versed with the game itself.
Also i would like to mention that i create the following scripts with one espress goal and after successfull execution will leave code snippits as comments behind

Now lets get started: in the folderstructure there is a place to put the save file, which is a .timber file. this also means that it is essentially a .zip file, wich can be unpacked quite easily.

this will result among other things in a file called "world.json", which is mainly what we will be editing... right after imports

In [1]:
import json
import re
import uuid


In [2]:
world_file = open("saves/world.json")
world = json.load(world_file)

print("world loaded")
print("Game Version ", world["GameVersion"])
print("Map Size ", world["Singletons"]["MapSize"]["Size"])

world loaded
Game Version  1.0.12.0-ab0d0fd-xsw
Map Size  {'X': 128, 'Y': 128}


now the savefile is loaded into a modifyable object from where we can read data and potentially add our own.
for my own purpose i will not mess with terrain. this also means that i will write this code for a gamefile without terrain in order to avoid complications

hence i will only care about adding enities (buildings) to the save file

In [3]:
entities = world["Entities"]

For the purpose of this world I would like to place a lot of Logic, connect them correctly and achieve a functional simulation entirely run through timberborn.
but to get there we will have to work in steps.
First lets add in all the scaffolding (Plattforms) on which the logic will be placed.
the file in its current state already has some of them, so we need to detect them and not try to generate any in their place

In [4]:
def is_platform(ent):
    return ent["Template"] == "DoublePlatform.IronTeeth"

def get_coords(ent):
    x = ent["Components"]["BlockObject"]["Coordinates"]["X"]
    y = ent["Components"]["BlockObject"]["Coordinates"]["Y"]
    z = ent["Components"]["BlockObject"]["Coordinates"]["Z"]
    
    return ({x,y,z})

In [5]:
existing_platforms = filter(is_platform, entities)

def has_platform(x, y, z):
    for e in existing_platforms:
        if (get_coords(e) == {x, y, z}):
            return (1)
    return (0)
    

In [6]:
def generate_platform(x, y, z):
    id = str(uuid.uuid4())
    
    return({"Id": id,"Template": "DoublePlatform.IronTeeth","Components": {"BlockObject": {"Coordinates": {"X": x,"Y": y,"Z": z},"Orientation": "Cw90"},"Inventory:ConstructionSite": {"Storage": {"Goods": [{"Good": "Plank","Amount": 8}]}}}})

lets try to fill the map with a layer of platforms

first we need to get the boundries fof the map

In [7]:
world_x =  world["Singletons"]["MapSize"]["Size"]["X"]
world_y =  world["Singletons"]["MapSize"]["Size"]["Y"]

# for x in range(world_x):
#     for y in range(world_y):
#         if (has_platform(x, y, 0)):
#             continue
#         new_platform = generate_platform(x, y, 0)
#         entities.append(new_platform)

p = generate_platform(50,50,0)
entities.append(p)

now to complie our local (and changed) world into a world file

In [8]:
out = open("saves/new_world.json", "w")
json.dump(world, out)